# Comparing London and Paris

Runs after `Airbnb_Project_Guided.ipynb` has been run for both cities. Two questions: do the predictors behave the same in both cities, and which findings of Biswas et al. (2020) and Sengupta et al. (2021) hold?

In [ ]:
# Setup: both cities' model data from the analysis notebook (run it for London and Paris first)
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from models import CORE, EXPECTED, replication

TABLES = "../outputs/tables"
data = {c: pd.read_csv(f"../data/{c}/{c}_model_data.csv.gz") for c in ["London", "Paris"]}
print({c: len(d) for c, d in data.items()})

## Pooled model with city interactions
Model A fitted on both cities, with a `predictor × paris` term for every predictor. A significant interaction means the effect differs between the cities.

In [ ]:
# Pooled Model A with a city interaction on every predictor
# Predictors are standardised within each city, so effects are per SD of the local market
Pool = pd.concat([d[["host_id", "number_of_reviews"] + CORE].assign(paris=int(c == "Paris")) for c, d in data.items()],
                 ignore_index=True)
pooled = smf.negativebinomial(f"number_of_reviews ~ ({' + '.join(CORE)}) * paris", Pool).fit(
    method="bfgs", maxiter=5000, disp=0, cov_type="cluster", cov_kwds={"groups": Pool["host_id"]})
assert pooled.mle_retvals["converged"], "Pooled model did not converge"

b, inter = pooled.params, [f"{v}:paris" for v in CORE]
cities = pd.DataFrame({
    "London IRR": np.exp(b[CORE]).values,
    "Paris IRR": np.exp(b[CORE].values + b[inter].values),
    "interaction p": pooled.pvalues[inter].values,
}, index=CORE)
cities["differs at 5%"] = cities["interaction p"] < .05
print(f"Pooled listings: {len(Pool):,}")
print(cities.round(3).to_string())
cities.to_csv(f"{TABLES}/pooled_interactions.csv")

## Replication scorecard
✅ significant in the direction the papers report, ❌ significant in the opposite direction, n.s. not significant at 5%. Directions: Biswas et al. (2020) Tables 4 to 7; Sengupta et al. (2021) Tables 5 to 8. Neither paper includes Paris; Biswas reports London as its own cluster (Cluster 4); Sengupta's nearest comparison is its Europe model.

In [ ]:
# Replication scorecard: our estimates against the direction reported by Biswas et al. (2020) and Sengupta et al. (2021)
# Core variables from Model A, text variables from Model A+, guest-side variables from Model B
def verdict(t, v, sign):
    if v not in t.index:
        return "not estimated"
    found = "+" if t.loc[v, "IRR"] > 1 else "-"
    if t.loc[v, "p"] >= .05:
        return f"n.s. ({t.loc[v, 'IRR']:.2f})"
    return f"{'✅' if found == sign else '❌'} {t.loc[v, 'IRR']:.2f}"

card = pd.DataFrame({"expected": {v: s for v, (s, _) in EXPECTED.items()}, "source": {v: src for v, (_, src) in EXPECTED.items()}})
for c, m in data.items():
    estimates = replication(m)
    card[c] = [verdict(estimates, v, s) for v, (s, _) in EXPECTED.items()]

untestable = pd.DataFrame({"expected": ["faster +", "+"], "source": ["Sengupta", "Both"], "London": "⚠️ empty in file", "Paris": "⚠️ empty in file"},
                          index=["host_response_time", "instant_bookable"])
card = pd.concat([card, untestable])
print(card.to_string())
card.to_csv(f"{TABLES}/scorecard.csv")